In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
conn = sqlite3.connect("../db.sqlite3")
orders = pd.read_sql_query("SELECT * FROM restaurant_order", conn)
order_item = pd.read_sql_query("SELECT * FROM restaurant_orderitem", conn)
product = pd.read_sql_query("SELECT * FROM restaurant_product", conn)
conn.close()
df = order_item.merge(product, left_on="product_id", right_on="id", suffixes=("", "_product"))
df = df.merge(orders, left_on="order_id", right_on="id", suffixes=("", "_order"))
df.to_csv("all_data.csv", index=False)
df = pd.read_csv("all_data.csv")
df["datetime"] = pd.to_datetime(df["datetime"])
df["Order Hour"] = df["datetime"].dt.hour
df["Order Day of the Week"] = df["datetime"].dt.day_name()
df["Item Price"] = df["price"] * df["quantity"]
df.head()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
product_sales = (df.groupby("name")["quantity"]).sum().sort_values(ascending=False)
top_10 = product_sales.head(10)
plt.figure(figsize=(8, 8))
plt.pie(top_10, labels=top_10.index, autopct="%1.1f%%", startangle=140, counterclock=False)
plt.title("Top 10")
plt.axis("equal")
plt.tight_layout()
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["price"] * df["quantity"]
revenue_by_product = df.groupby("name")["item_price"].sum().sort_values(ascending=False)
top_10_revenue = revenue_by_product.head(10)
plt.figure(figsize=(8, 8))
plt.pie(top_10_revenue, labels=top_10_revenue.index, autopct="%1.1f%%", startangle=140, counterclock=False)
plt.title("Top 10")
plt.axis("equal")
plt.tight_layout()
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
income_by_hour = df.groupby("Order Hour")["Item Price"].sum()
plt.figure(figsize=(12, 6))
income_by_hour.plot(kind="bar", color="green")
plt.title("Total Restaurant Income by Order Hour")
plt.xlabel("Order Hour")
plt.ylabel("Total Income")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
income_by_day = df.groupby("Order Day of the Week")["Item Price"].sum()
ordered_days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
income_by_day = income_by_day.reindex(ordered_days)
plt.figure(figsize=(12, 6))
income_by_day.plot(kind="bar", color="green")
plt.title("Total Restaurant Income by Order Day of the Week")
plt.xlabel("Order Day of the Week")
plt.ylabel("Total Income")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()